# Building AI Workflows with LangChain: Prompts, Chains, and LLM Integration



<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/langchain-intro-202503.ipynb
</div>

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>

## Environment Setup

<br>

> 💡 **Important:**
>
> This notebook requires **LangChain < 0.3**.
>
> Below, you will find two options for installing the required dependencies. Choose the one that best matches your environment::
>
> 🖥️ **Running locally?**  
> → Use **Option A** to create a dedicated virtual environment (recommended).
>
> ☁️ **Using Google Colab or want a quick setup?**  
> → Use **Option B** to install the required dependencies directly.
>

<br><br>


### Option A: Use a virtual environment

Open a terminal and run the following commands.


<br>

**1. Create a virtual environment:**

```bash
python -m venv .venv/langchain-v0.2.x
```

<br>

**2. Activate the virtual environment:**

- **macOS / Linux:**
```bash
    source .venv/langchain-v0.2.x/bin/activate
```

- **Windows (PowerShell):**
```powershell
    .\.venv\langchain-v0.2.x\Scripts\Activate.ps1
```

<br>

**3. Install the required packages:**

```bash
python -m pip install \
    "langchain<0.3" \
    "langchain-core<0.3" \
    "langchain-community<0.3" \
    "langchain-openai<0.2" \
    ipykernel
```

<br>

**4. Register the environment as a Jupyter kernel:**

```bash
python -m ipykernel install \
    --user \
    --name langchain-v0.2.x \
    --display-name "Python (LangChain 0.2.x)"
```

<br>

**5. Select the correct kernel:**

Once you've completed the previous steps, do the following:
1. Open this notebook in your favourite environment (e.g., Jupyter or VS Code)
2. Select the Kernel you've just created
    - **Jupyter**: Kernel → Change Kernel → Python (LangChain 0.2.x).
    - **VS Code**: Click the Kernel selector in the top-right corner of the notebook editor → Jupyter Kernel → Python (LangChain 0.2.x)
        - Notice that you need to select "Jupyter Kernel" (not "Python Environments")
        - If "Python (LangChain 0.2.x)" doesn't appear in the kernel list, reload VS Code:
            - Press Cmd + Shift + P to open the Command Palette
            - Type Developer: Reload Window and press Enter
            - Try selecting the kernel again
3. Run the notebook as usual.

<br>

> **Notes:**
>
> - Make sure to add the directory `.venv` to your `.gitignore`
> - The virtual environment setup only needs to be completed once. The environment can then be reused for other notebooks that require **LangChain < 0.3**, without affecting your default Python environment or newer LangChain installations.

<br>

### Option B: Install dependencies directly

If you are using Google Colab, or you're having problems configuring a virtual environment, create a code cell and run the command below:

```python
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"
```

<br>


</div>

<br>

## Check LangChain version

For this notebook, you'll need LangChain 0.2.x (e.g., 0.2.17)

To check your LangChain version, you can run the command below:

In [10]:
!pip show langchain

Name: langchain
Version: 0.2.17
Summary: Building applications with LLMs through composability
Home-page: https://github.com/langchain-ai/langchain
Author: 
Author-email: 
License: MIT
Location: /Users/luis/Desktop/ironhack_june26/1_ai_eng_lectures/.venv/langchain-v0.2.x/lib/python3.11/site-packages
Requires: aiohttp, langchain-core, langchain-text-splitters, langsmith, numpy, pydantic, PyYAML, requests, SQLAlchemy, tenacity
Required-by: langchain-community


<br>

## Install other dependencies we'll use in this notebook


In [11]:
!pip install dotenv numexpr


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


<br>

## Generating Text

In [12]:
import warnings
warnings.filterwarnings("ignore")
from langchain_openai import OpenAI
from langchain import HuggingFaceHub
from langchain_core.runnables import Runnable
from langchain_core.prompts import PromptTemplate
from langchain.chains import SimpleSequentialChain, LLMChain, LLMMathChain, TransformChain, SequentialChain
from langchain_core.output_parsers import StrOutputParser
from langchain import FewShotPromptTemplate
from langchain.callbacks import get_openai_callback
import inspect
import re
import os


from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [13]:
llm = OpenAI(model_name="gpt-3.5-turbo-instruct", api_key=OPENAI_API_KEY, temperature=0.7)

<br>

### Prompts and Prompt templates ✏️

A **prompt** is simply the textual instruction we give a model to get a specific output.

Imagine we want an outline about tennis. Our prompt could be: *"Write me an outline on Tennis"*. Now suppose we want the same kind of outline, but for a different sport, like cricket. The naive approach would be to manually rewrite the whole prompt every time the sport changes.

This doesn't scale well: hardcoding a new prompt for every possible input is repetitive and hard to maintain. Instead, we want a single reusable prompt with a "placeholder" for whatever value the user provides.

This is exactly what **prompt templates** solve. A prompt template is a reusable prompt with one or more placeholders (input variables) that get filled in dynamically at runtime. Instead of rewriting the whole prompt, we just swap out the variable.


<br>

## Structure of a Prompt

A prompt can consist of multiple components:

* Instructions
* External information or context
* User input or query
* Output indicator

Not all prompts require all of these components, but often a good prompt will use two or more of them. Let's define what they all are more precisely.

- **Instructions** tell the model what to do, typically how it should use inputs and/or external information to produce the output we want.

- **External information or context** are additional information that we either manually insert into the prompt, retrieve via a vector database (long-term memory), or pull in through other means (API calls, calculations, etc).

- **User input or query** is typically a query directly input by the user of the system.

- **Output indicator** is the *beginning* of the generated text. For a model generating Python code we may put `import ` (as most Python scripts begin with a library `import`), or a chatbot may begin with `Chatbot: ` (assuming we format the chatbot script as lines of interchanging text between `User` and `Chatbot`).

Each of these components should usually be placed in the order we've described them. We start with instructions, provide context (if needed), then add the user input, and finally end with the output indicator.

<br>

Let's see two different ways to create a PromptTemplate object:

In [14]:
# Option 1: `from_template()` — input_variables are inferred automatically
# from the placeholders in the template string.

template = PromptTemplate.from_template("Write me an outline on {input_parameter}?")   
user_input = input("Enter sport : ")
prompt = template.format(input_parameter=user_input)
print("Prompt :",prompt)

Prompt : Write me an outline on tennis?


In [15]:
# Option 2: Explicit constructor — you declare input_variables yourself.

prompt = PromptTemplate(input_variables=["input_parameter"], template="Write me an outline on {input_parameter}")
print("Prompt :",prompt)

Prompt : input_variables=['input_parameter'] template='Write me an outline on {input_parameter}'


We wouldn't typically know what the users prompt is beforehand, so we actually want to add this in. So rather than writing the prompt directly, we create a PromptTemplate with a single input variable query.

<br>

## Chains

We now move on to another core LangChain concept: **Chains**. A chain is responsible for the data flow inside LangChain — it connects the different pieces (prompt template, LLM, etc.) together so that data flows automatically from one step to the next. LangChain supports many different types of chains; we'll explore several of them later.

The simplest one is `LLMChain`. It takes the prompt template we created above, fills it in with our dynamic input, and passes the resulting prompt to the LLM. Let's define one below.

In [16]:
from langchain.schema.runnable import RunnableSequence

chain = prompt | llm 


Now that we have created a prompt template and a chain we can now input any topic we want. Instead of topic "Tennis" we can input "Cricket" or any other topic of your choice

In [17]:
result = chain.invoke({"input_parameter": "Cricket"})
print(result)




I. Introduction
    A. Definition of cricket
    B. History and origins of cricket
    C. Popularity of cricket worldwide

II. Objectives and Rules of the Game
    A. Objective of the game
    B. Basic rules of cricket
    C. Scoring system
    D. Playing equipment

III. Playing Field and Positions
    A. Description of the playing field
    B. Different sections of the field
    C. Positions and roles of players
    D. Fielding techniques and strategies

IV. Types of Matches
    A. Test cricket
    B. One Day International (ODI) cricket
    C. Twenty20 (T20) cricket
    D. Differences between the three types of matches

V. The Batting Team
    A. Roles and responsibilities of batting team players
    B. Strategies for scoring runs
    C. Techniques for batting
    D. Ways to get out in cricket

VI. The Bowling Team
    A. Roles and responsibilities of bowling team players
    B. Types of bowling techniques
    C. Strategies for getting wickets
    D. Rules for bowling in cricket

VI

<br>

Now let's extend it for a multi-input prompt. Let's generate an introductory paragraph to a blog post with variables title, audience and tone of voice

In [18]:
prompt = PromptTemplate(
    input_variables=["title", "audience", "tone"],
    template="""This program will generate an introductory paragraph to a blog post given a blog title, audience, and tone of voice

    Blog Title: {title}
    Audience: {audience}
    Tone of Voice: {tone}""",
)
chain = LLMChain(llm=llm, prompt=prompt)

In [19]:
print(chain.run(title="Best Activities in Toronto", audience="Millenials", tone="Lighthearted"))



Welcome to the ultimate guide for the adventurous and fun-loving millenials out there! Toronto, the bustling city in Canada, offers a plethora of activities that are perfect for the young and restless. From exploring the vibrant street art scene to indulging in delicious food and drinks, this city has something for everyone. So grab your friends, put on your walking shoes, and get ready to discover the best activities Toronto has to offer. Let's dive in and make some unforgettable memories together!


<br>
<hr>
<br>

## Practice: Prompt Templates and Chains

Instructions:
- https://gist.github.com/luisjunco/df636313b0ced004033e85655989d1f4


Time: 15 min.

<br>
<hr>
<br>

<br>

## Combining Chains

We often need to perform multiple steps with an LLM, where the output of one step feeds into the next. For example, generating an outline for a topic, and then using that outline to write a full blog article. Manually copying the output of the first step into the second would work, but it's tedious and error-prone.

Instead, we can combine chains so this happens automatically in a single step. We can do this using a `SequentialChain`, which takes the output of one chain and passes it as the input to the next.

In [20]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Write a brief paragraph on this topic: {topic}",
)

llm = OpenAI(temperature=0.9, max_tokens=-1)

chain_one = LLMChain(llm=llm, prompt=prompt)

second_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Translate the given text to Spanish 

    Text:
    {text}""",
)
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [ ]:
#
# Create SimpleSequentialChain
# - Note: we're using verbose=True to be able to see the intermidiate steps
#
overall_chain = SimpleSequentialChain(chains=[chain_one, chain_two], verbose=True)

# Run the chain specifying only the input variable for the first chain.
result = overall_chain.run("Tennis")
print(result)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")




Tennis is a popular sport around the world that is played by people of all ages and skill levels. It is a racket sport that can be played individually against a single opponent (singles) or between two teams of two players each (doubles). The objective of the game is to hit a small ball with a racket over a net and into the opponent's court, with the goal of making it difficult for the opponent to return the shot. Tennis requires excellent hand-eye coordination, speed, agility, and strategy. It is not only a physically demanding sport, but also a mentally challenging one. As a result, it provides a great opportunity for exercise, competition, and social interaction. 


El tenis es un deporte popular en todo el mundo que es jugado por personas de todas las edades y niveles de habilidad. Es un deporte de raqueta que puede ser jugado individualmente contra un solo oponente (individuales) o entre dos equipos de dos jugadores cada uno (dobles). El objetivo del juego es golpear una pelota 

<br>

## Revisit the Prompt in Langchain

The prompt template classes in Langchain are built to make constructing prompts with dynamic inputs easier. Of these classes, the simplest is the PromptTemplate. We’ll test this by adding a single dynamic input to our previous prompt, the user query.

In [22]:
template = """Answer the question based on the context below. If the
question cannot be answered using the information provided answer
with "I don't know".

Context: Large Language Models (LLMs) are the latest models used in NLP.
Their superior performance over smaller models has made them incredibly
useful for developers building NLP enabled applications. These models
can be accessed via Hugging Face's `transformers` library, via OpenAI
using the `openai` library, and via Cohere using the `cohere` library.

Question: {query}

Answer: """

prompt_template = PromptTemplate(
    input_variables=["query"],
    template=template
)

<br>

With this, we can use the **format** method on our **prompt_template** to see the effect of passing a query to the template.

In [23]:
print(
    prompt_template.format(
        query="Which libraries and model providers offer LLMs?"
    )
)

Answer the question based on the context below. If the
question cannot be answered using the information provided answer
with "I don't know".

Context: Large Language Models (LLMs) are the latest models used in NLP.
Their superior performance over smaller models has made them incredibly
useful for developers building NLP enabled applications. These models
can be accessed via Hugging Face's `transformers` library, via OpenAI
using the `openai` library, and via Cohere using the `cohere` library.

Question: Which libraries and model providers offer LLMs?

Answer: 


In [24]:
prompt = prompt_template.format(query="Which libraries and model providers offer LLMs?")
chain = prompt_template | llm
chain.invoke(prompt)

"Hugging Face's `transformers` library, OpenAI's `openai` library, and Cohere's `cohere` library."

<br>

### Few Shot Prompt Templates

The success of LLMs comes from their large size and ability to store “knowledge” within the model parameter, which is learned during model training. However, there are more ways to pass knowledge to an LLM. The two primary methods are:

- Parametric knowledge — the knowledge mentioned above is anything that has been learned by the model during training time and is stored within the model weights (or parameters).
- Source knowledge — any knowledge provided to the model at inference time via the input prompt.
Langchain’s FewShotPromptTemplate caters to source knowledge input. The idea is to “train” the model on a few examples — we call this few-shot learning — and these examples are given to the model within the prompt.

Few-shot learning is perfect when our model needs help understanding what we’re asking it to do. We can see this in the following example:

In [25]:
# create our examples
examples = [ #you can have n such example of query-answer pairs
    {
        "query": "How are you?",
        "answer": "I can't complain but sometimes I still do."
    }, {
        "query": "What time is it?",
        "answer": "It's time to get a watch."
    }
]

# create a example template
example_template = """
User: {query}
AI: {answer}
"""

# create a prompt example from above template
example_prompt = PromptTemplate(
    input_variables=["query", "answer"],
    template=example_template
)

# now break our previous prompt into a prefix and suffix
# the prefix is our instructions
prefix = """The following are exerpts from conversations with an AI
assistant. The assistant is typically sarcastic and witty, producing
creative  and funny responses to the users questions. Here are some
examples: 
"""
# and the suffix our user input and output indicator
suffix = """
User: {query}
AI: """

# now create the few shot prompt template
few_shot_prompt_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["query"],
    example_separator="\n\n"
)

<br>

If we then pass in the examples and user query, we will get this:

In [26]:
query = "What is the meaning of life?"

print(few_shot_prompt_template.format(query=query))

The following are exerpts from conversations with an AI
assistant. The assistant is typically sarcastic and witty, producing
creative  and funny responses to the users questions. Here are some
examples: 



User: How are you?
AI: I can't complain but sometimes I still do.



User: What time is it?
AI: It's time to get a watch.



User: What is the meaning of life?
AI: 


<br>

Considering this, we need to balance the number of examples included and our prompt size. Our hard limit is the maximum context size, but we must also consider the cost of processing more tokens through the LLM. Fewer tokens mean a cheaper service and faster completions from the LLM.

The `FewShotPromptTemplate` allows us to vary the number of examples included based on these variables. First, we create a more extensive list of examples:

In [27]:
examples = [
    {
        "query": "How are you?",
        "answer": "I can't complain but sometimes I still do."
    }, {
        "query": "What time is it?",
        "answer": "It's time to get a watch."
    }, {
        "query": "What is the meaning of life?",
        "answer": "42"
    }, {
        "query": "What is the weather like today?",
        "answer": "Cloudy with a chance of memes."
    }, {
        "query": "What is your favorite movie?",
        "answer": "Terminator"
    }, {
        "query": "Who is your best friend?",
        "answer": "Siri. We have spirited debates about the meaning of life."
    }, {
        "query": "What should I do today?",
        "answer": "Stop talking to chatbots on the internet and go outside."
    }
]

<br>

After this, rather than passing the examples directly, we actually use a `LengthBasedExampleSelector` like so:

In [28]:
from langchain.prompts.example_selector import LengthBasedExampleSelector

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=50  # this sets the max length that examples should be
)

<br>

We then pass our `example_selector` to the `FewShotPromptTemplate` to create a new — and dynamic — prompt template:

In [29]:
# now create the few shot prompt template
dynamic_prompt_template = FewShotPromptTemplate(
    example_selector=example_selector,  # use example_selector instead of examples
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["query"],
    example_separator="\n"
)

<br>

Now if we pass a shorter or longer query, we should see that the number of included examples will vary.

In [30]:
print(dynamic_prompt_template.format(query="How do birds fly?"))

The following are exerpts from conversations with an AI
assistant. The assistant is typically sarcastic and witty, producing
creative  and funny responses to the users questions. Here are some
examples: 


User: How are you?
AI: I can't complain but sometimes I still do.


User: What time is it?
AI: It's time to get a watch.


User: What is the meaning of life?
AI: 42


User: How do birds fly?
AI: 


<br>

Passing a longer question will result in fewer examples being included:

In [31]:
query = """If I am in America, and I want to call someone in another country, I'm
thinking maybe Europe, possibly western Europe like France, Germany, or the UK,
what is the best way to do that?"""

print(dynamic_prompt_template.format(query=query))

The following are exerpts from conversations with an AI
assistant. The assistant is typically sarcastic and witty, producing
creative  and funny responses to the users questions. Here are some
examples: 


User: How are you?
AI: I can't complain but sometimes I still do.


User: If I am in America, and I want to call someone in another country, I'm
thinking maybe Europe, possibly western Europe like France, Germany, or the UK,
what is the best way to do that?
AI: 


<br>

With this, we’re returning fewer examples within the prompt variable. Allowing us to limit excessive token usage and avoid errors from surpassing the maximum context window of the LLM.

An extra utility we will use is this function that will tell us how many tokens we are using in each call. This is a good practice that is increasingly important as we use more complex tools that might make several calls to the API (like agents). It is very important to have a close control of how many tokens we are spending to avoid unsuspected expenditures.

<br>

## Let's revisit chains

In [32]:
def count_tokens(chain, query):
    with get_openai_callback() as cb:
        result = chain.run(query)
        print(f'Spent a total of {cb.total_tokens} tokens')

    return result

<br>

Chains are divided in three types: Utility chains, Generic chains and Combine Documents chains. In this edition, we will focus on the first two since the third is too specific (will be covered in due course).

1. Utility Chains: chains that are usually used to extract a specific answer from a llm with a very narrow purpose and are ready to be used out of the box.
2. Generic Chains: chains that are used as building blocks for other chains but cannot be used out of the box on their own.

<br>

### Utility Chains

Let's start with a simple utility chain. The `LLMMathChain` gives llms the ability to do math. Let's see how it works!

In [33]:
llm_math = LLMMathChain.from_llm(llm=llm)


count_tokens(llm_math, "What is 13 raised to the .3432 power?")

Spent a total of 230 tokens


'Answer: 2.4116004626599237'

<br>

Let's see what is going on here. The chain recieved a question in natural language and sent it to the llm. The llm returned a Python code which the chain compiled to give us an answer. A few questions arise.. How did the llm know that we wanted it to return Python code? 

**Enter prompts**

The question we send as input to the chain is not the only input that the llm recieves 😉. The input is inserted into a wider context, which gives precise instructions on how to interpret the input we send. This is called a _prompt_. Let's see what this chain's prompt is!

In [34]:
print(llm_math.prompt.template)

Translate a math problem into a expression that can be executed using Python's numexpr library. Use the output of running this code to answer the question.

Question: ${{Question with math problem.}}
```text
${{single line mathematical expression that solves the problem}}
```
...numexpr.evaluate(text)...
```output
${{Output of running the code}}
```
Answer: ${{Answer}}

Begin.

Question: What is 37593 * 67?
```text
37593 * 67
```
...numexpr.evaluate("37593 * 67")...
```output
2518731
```
Answer: 2518731

Question: 37593^(1/5)
```text
37593**(1/5)
```
...numexpr.evaluate("37593**(1/5)")...
```output
8.222831614237718
```
Answer: 8.222831614237718

Question: {question}



<br>

Ok.. let's see what we got here. So, we are literally telling the llm that for complex math problems **it should not try to do math on its own** but rather it should print a Python code that will calculate the math problem instead. Probably, if we just sent the query without any context, the llm would try (and fail) to calculate this on its own. Wait! This is testable.. let's try it out! 🧐

In [35]:
# we set the prompt to only have the question we ask
prompt = PromptTemplate(input_variables=['question'], template='{question}')
llm_chain = LLMChain(prompt=prompt, llm=llm)

# we ask the llm for the answer with no context

count_tokens(llm_chain, "What is 13 raised to the .3432 power?")

Spent a total of 29 tokens


'\n\n13 raised to the .3432 power is approximately 3.1871.'

<br>

Wrong answer! Herein lies the power of prompting and one of our most important insights so far: 

**Insight**: _by using prompts intelligently, we can force the llm to avoid common pitfalls by explicitly and purposefully programming it to behave in a certain way._

Another interesting point about this chain is that it not only runs an input through the llm but it later compiles Python code. Let's see exactly how this works.

In [36]:
print(inspect.getsource(llm_math._call))

    def _call(
        self,
        inputs: Dict[str, str],
        run_manager: Optional[CallbackManagerForChainRun] = None,
    ) -> Dict[str, str]:
        _run_manager = run_manager or CallbackManagerForChainRun.get_noop_manager()
        _run_manager.on_text(inputs[self.input_key])
        llm_output = self.llm_chain.predict(
            question=inputs[self.input_key],
            stop=["```output"],
            callbacks=_run_manager.get_child(),
        )
        return self._process_llm_result(llm_output, _run_manager)



<br>

So we can see here that if the llm returns Python code we will compile it with a Python REPL* simulator. We now have the full picture of the chain: either the llm returns an answer (for simple math problems) or it returns Python code which we compile for an exact answer to harder problems. Smart!

Also notice that here we get our first example of **chain composition**, a key concept behind what makes langchain special. We are using the `LLMMathChain` which in turn initializes and uses an `LLMChain` (a 'Generic Chain') when called. We can make any arbitrary number of such compositions, effectively 'chaining' many such chains to achieve highly complex and customizable behaviour.

Utility chains usually follow this same basic structure: there is a prompt for constraining the llm to return a very specific type of response from a given query. We can ask the llm to create SQL queries, API calls and even create Bash commands on the fly 🔥

The list continues to grow as langchain becomes more and more flexible and powerful so we encourage you to [check it out](https://python.langchain.com/v0.2/docs/how_to/) and tinker with the example notebooks that you might find interesting.

*_A Python REPL (Read-Eval-Print Loop) is an interactive shell for executing Python code line by line_

<br>

### Generic chains

There are only three Generic Chains in langchain and we will go all in to showcase them all in the same example. Let's go!

Say we have had experience of getting dirty input texts. Specifically, as we know, llms charge us by the number of tokens we use and we are not happy to pay extra when the input has extra characters. Plus its not neat 😉

First, we will build a custom transform function to clean the spacing of our texts. We will then use this function to build a chain where we input our text and we expect a clean text as output.

In [37]:
def transform_func(inputs: dict) -> dict:
    text = inputs["text"]
    
    # replace multiple new lines and multiple spaces with a single one
    text = re.sub(r'(\r\n|\r|\n){2,}', r'\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    return {"output_text": text}

<br>

Importantly, when we initialize the chain we do not send an llm as an argument. As you can imagine, not having an llm makes this chain's abilities much weaker than the example we saw earlier. However, as we will see next, combining this chain with other chains can give us highly desirable results.

In [38]:
clean_extra_spaces_chain = TransformChain(input_variables=["text"], output_variables=["output_text"], transform=transform_func)

In [39]:
clean_extra_spaces_chain.run('A random text  with   some irregular spacing.\n\n\n     Another one   here as well.')

'A random text with some irregular spacing.\n Another one here as well.'

<br>

Great! Now things will get interesting.

Say we want to use our chain to clean an input text and then paraphrase the input in a specific style, say a poet or a policeman. As we now know, the `TransformChain` does not use a llm so the styling will have to be done elsewhere. That's where our `LLMChain` comes in. We know about this chain already and we know that we can do cool things with smart prompting so let's take a chance!

First we will build the prompt template:

In [40]:
template = """Paraphrase this text:

{output_text}

In the style of a {style}.

Paraphrase: """
prompt = PromptTemplate(input_variables=["style", "output_text"], template=template)

<br>

And next, initialize our chain:

In [41]:
style_paraphrase_chain = LLMChain(llm=llm, prompt=prompt, output_key='final_output')

<br>

Great! Notice that the input text in the template is called 'output_text'. Can you guess why?

We are going to pass the output of the `TransformChain` to the `LLMChain`!

Finally, we need to combine them both to work as one integrated chain. For that we will use `SequentialChain` which is our third generic chain building block.

In [42]:
sequential_chain = SequentialChain(chains=[clean_extra_spaces_chain, style_paraphrase_chain], input_variables=['text', 'style'], output_variables=['final_output'])

<br>

Our input is the langchain docs description of what chains are but dirty with some extra spaces all around.

In [43]:
input_text = """
Chains allow us to combine multiple 


components together to create a single, coherent application. 

For example, we can create a chain that takes user input,       format it with a PromptTemplate, 

and then passes the formatted response to an LLM. We can build more complex chains by combining     multiple chains together, or by 


combining chains with other components.
"""

<br>

We are all set. Time to get creative!

In [44]:
count_tokens(sequential_chain, {'text': input_text, 'style': 'a 90s rapper'})

Spent a total of 160 tokens


"Chains be the key to unitin' different parts to make one dope app. Like, we can make a chain that takes in user words, lays down a PromptTemplate, and then gives it to an LLM. We can even make even fresher chains by mixin' them with other chains or other dope components."

<br>

# LangChain's Chains Operations

In LangChain, chains are a core concept used to combine different operations in a sequential or conditional manner. These chains allow you to build complex workflows by linking various components together. Here are some of the main types of chain operations and how they can be used:

<br>

## 1. SequentialChain

`SequentialChain` is used to link multiple components together so that the output of one component becomes the input for the next. This is useful for creating multi-step processes.

In [45]:
from langchain.chains import SequentialChain, LLMChain
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI  

# Define individual components
prompt1 = PromptTemplate(template="Translate English to French: {text}", input_variables=["text"])
prompt2 = PromptTemplate(template="Translate French to Spanish: {french_text}", input_variables=["french_text"])

# Use the updated ChatOpenAI class
llm = ChatOpenAI(model_name="gpt-4-turbo")


# Create individual LLMChains
chain1 = LLMChain(llm=llm, prompt=prompt1, output_key="french_text")
chain2 = LLMChain(llm=llm, prompt=prompt2, output_key="spanish_text")

# Create a SequentialChain
chain = SequentialChain(
    chains=[chain1, chain2],
    input_variables=["text"],
    output_variables=["spanish_text"]  # Final output variable
)

# Run the chain
result = chain({"text": "The house is wonderful."})
print(result["spanish_text"])


La casa es maravillosa.


<br>

## 2. LLMChain
This is one of the most commonly used chains. It's a chain that combines a language model (LLM) with a prompt template.  It is used when you want to execute a single step, such as summarization, translation, or Q&A.


In [46]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI

# Define the prompt template
prompt = PromptTemplate(template="Summarize the following text in 10 words or fewer: {text}", input_variables=["text"])

# Initialize the LLM
llm = ChatOpenAI(model_name="gpt-4-turbo")

# Create the LLMChain
chain = LLMChain(llm=llm, prompt=prompt)

# Run the chain
#result = chain.run({"text": "LangChain is a powerful library for building language model chains."})
text = """
LangChain is an open-source framework designed to facilitate the development of applications using large language models (LLMs). 
It provides tools to integrate various components such as prompt templates, memory, chains, and agents. 
LangChain enables developers to build chatbots, question-answering systems, and other AI-powered applications efficiently.
"""

result = chain.invoke({"text": text})
print(result["text"])

LangChain: Open-source framework for developing LLM-based applications.


<br>

## 3. LLMRouterChain

`LLMRouterChain` helps route queries to the most appropriate large language model (LLM) or tool based on the input. It is particularly useful when working with multiple models or APIs with different capabilities.

In [47]:
from langchain.chains.router.multi_prompt import MULTI_PROMPT_ROUTER_TEMPLATE

destinations = """
animals: prompt for animal expert
vegetables: prompt for a vegetable expert
"""

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=destinations)

print(router_template.replace("`", "'"))  # for rendering purposes


from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

router_prompt = PromptTemplate(
    # Note: here we use the prompt template from above. Generally this would need
    # to be customized.
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

chain = LLMRouterChain.from_llm(llm, router_prompt)



Given a raw text input to a language model select the model prompt best suited for the input. You will be given the names of the available prompts and a description of what the prompt is best suited for. You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
'''json
{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}
'''

REMEMBER: "destination" MUST be one of the candidate prompt names specified below OR it can be "DEFAULT" if the input is not well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>

animals: prompt for animal expert
vegetables: prompt for a vegetable expert


<< INPUT >>
{input}

<

In [48]:
result = chain.invoke({"input": "What color are carrots?"})

print(result["destination"])

vegetables


<br>

## 4. RunnableParallel
`RunnableParallel` RunnableParallel allows running multiple chains, functions, or callables in parallel and returns a dictionary of results. It is part of the Runnable framework, which provides more flexibility and better performance.

In [49]:
from langchain_openai import ChatOpenAI  
from langchain.schema.runnable import RunnableParallel
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Define LLM
llm = ChatOpenAI(model="gpt-4")  

# Create prompts
prompt_1 = PromptTemplate(input_variables=["question"], template="Translate this to French: {question}")
prompt_2 = PromptTemplate(input_variables=["question"], template="Translate this to Spanish: {question}")

# Define two LLMChains
chain_1 = LLMChain(llm=llm, prompt=prompt_1)
chain_2 = LLMChain(llm=llm, prompt=prompt_2)

# Run both chains in parallel
parallel_chain = RunnableParallel(french=chain_1, spanish=chain_2)

# Example input
result = parallel_chain.invoke({"question": "Hello, how are you?"})

print(result)


{'french': {'question': 'Hello, how are you?', 'text': 'Bonjour, comment ça va?'}, 'spanish': {'question': 'Hello, how are you?', 'text': 'Hola, ¿cómo estás?'}}


<br>

## 5. MapReduceDocumentsChain
`MapReduceChain` is a chain in LangChain used for processing large numbers of documents efficiently. It applies a map-reduce approach, where:

1- Map Step: Each document is processed independently, generating intermediate responses.

2- Reduce Step: These intermediate responses are aggregated to produce the final output.

This chain is useful for summarization, QA over large documents, and other NLP tasks requiring multi-document processing.

In [50]:
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains.llm import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model

documents = [
    Document(page_content="Apples are red", metadata={"title": "apple_book"}),
    Document(page_content="Blueberries are blue", metadata={"title": "blueberry_book"}),
    Document(page_content="Bananas are yelow", metadata={"title": "banana_book"}),
]

llm = init_chat_model("gpt-4o-mini", model_provider="openai")


# Map
map_template = "Write a concise summary of the following: {docs}."
map_prompt = ChatPromptTemplate([("human", map_template)])
map_chain = LLMChain(llm=llm, prompt=map_prompt)


# Reduce
reduce_template = """
The following is a set of summaries:
{docs}
Take these and distill it into a final, consolidated summary
of the main themes.
"""
reduce_prompt = ChatPromptTemplate([("human", reduce_template)])
reduce_chain = LLMChain(llm=llm, prompt=reduce_prompt)


# Takes a list of documents, combines them into a single string, and passes this to an LLMChain
combine_documents_chain = StuffDocumentsChain(
    llm_chain=reduce_chain, document_variable_name="docs"
)

# Combines and iteratively reduces the mapped documents
reduce_documents_chain = ReduceDocumentsChain(
    # This is final chain that is called.
    combine_documents_chain=combine_documents_chain,
    # If documents exceed context for `StuffDocumentsChain`
    collapse_documents_chain=combine_documents_chain,
    # The maximum number of tokens to group documents into.
    token_max=1000,
)

# Combining documents by mapping a chain over them, then combining results
map_reduce_chain = MapReduceDocumentsChain(
    # Map chain
    llm_chain=map_chain,
    # Reduce chain
    reduce_documents_chain=reduce_documents_chain,
    # The variable name in the llm_chain to put the documents in
    document_variable_name="docs",
    # Return the results of the map steps in the output
    return_intermediate_steps=False,
)

In [51]:
result = map_reduce_chain.invoke(documents)

print(result["output_text"])

Fruits can be categorized by their distinctive colors, such as apples being red, blueberries being blue, and bananas being yellow.


<br>

These are some of the primary chain operations available in LangChain. They can be combined and customized to create complex workflows tailored to specific language processing tasks. Each chain type serves a different purpose and can be selected based on the requirements of the task at hand.